# Step 1: Data Scraping

In [3]:
from __future__ import annotations
 
import logging
import time
from collections import defaultdict
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
from typing import Callable, Mapping, Protocol, Sequence
 
import pandas as pd
import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
 
log = logging.getLogger(__name__)

## 1.1 Config

In [3]:
@dataclass(frozen=True)
class TableSpec:
    """One table we want from a page. `attrs` may contain a {label} placeholder."""
 
    name: str
    attrs: Mapping[str, str]
 
    def resolve(self, label: str) -> dict[str, str]:
        """{'id': 'results{label}91_overall'} + '2024-2025' -> {'id': 'results2024-202591_overall'}"""
        return {k: v.format(label=label) for k, v in self.attrs.items()}

In [4]:
@dataclass(frozen=True)
class SiteConfig:
    """Everything that differs from one website to another."""
 
    tables: tuple[TableSpec, ...]
    prepare_html: Callable[[str], str] = lambda html: html  # e.g. `uncomment`

In [5]:
def attrs_to_css(attrs: Mapping[str, str]) -> str:
    """{'id': 'stats_squads_standard_for'} -> 'table[id="stats_squads_standard_for"]'"""
    return "table" + "".join(f'[{k}="{v}"]' for k, v in attrs.items())

In [2]:
driver = uc.Chrome()

## Setting up Helper Functions

In [4]:
def create_urls(start_year: int, end_year:int):
    seasons = [f"{year}-{year+1}" for year in range(start_year, end_year)]

    urls = []
    
    for season in seasons:
        urls.append(f"https://fbref.com/en/comps/9/{season}/{season}-Premier-League-Stats")

    return urls, seasons

In [5]:
def resolve_attrs(attrs: dict[str, str], season: str) -> dict[str, str]:
    """{'id': 'results{season}91_overall'} + '2024-2025' -> {'id': 'results2024-202591_overall'}"""
    return {k: v.format(season=season) for k, v in attrs.items()}

In [6]:
def attrs_to_css(attrs: dict[str, str]) -> str:
    """{'id': 'stats_squads_standard_for'} -> 'table[id="stats_squads_standard_for"]'"""
    return "table" + "".join(f'[{k}="{v}"]' for k, v in attrs.items())

In [4]:
def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse MultiIndex columns into single-level names like 'Performance_Gls'."""
    if not isinstance(df.columns, pd.MultiIndex):
        return df

    flat = []
    for top, bottom in df.columns:
        top, bottom = str(top), str(bottom)
        if "Unnamed" in top or top == "" or top == bottom:
            flat.append(bottom)
        else:
            flat.append(f"{top}_{bottom}")

    df = df.copy()
    df.columns = flat
    return df

In [5]:
def scrape_page_tables(
    driver,
    url: str,
    tables: dict[str, dict[str, str]],
    season: str,
    page_timeout: int = 60,
    table_timeout: int = 10,
    settle: float = 3,
    uncomment: bool = False,
) -> tuple[dict[str, pd.DataFrame], list[str]]:
    tables = {name: resolve_attrs(a, season) for name, a in tables.items()}
    """Return (found tables, names of tables that were missing on this page)."""
    driver.get(url)

    # 1) The page itself must load (at least one table). If not, this raises.
    WebDriverWait(driver, page_timeout).until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    # 2) Give each wanted table a short chance to appear; a missing one is not fatal.
    for attrs in tables.values():
        try:
            WebDriverWait(driver, table_timeout).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, attrs_to_css(attrs)))
            )
        except TimeoutException:
            pass  # may not exist for this season; the parse step decides

    time.sleep(settle)
    html = driver.page_source
    if uncomment:  # for sites that hide tables inside <!-- ... -->
        html = html.replace("<!--", "").replace("-->", "")

    # 3) Parse each table independently.
    found, missing = {}, []
    for name, attrs in tables.items():
        try:
            found[name] = flatten_columns(pd.read_html(StringIO(html), attrs=attrs)[0])
        except ValueError:  # read_html: "No tables found"
            missing.append(name)

    return found, missing  


In [6]:
def scrape_seasons(
    driver,
    season_urls: dict[str, str],
    tables: dict[str, dict[str, str]],
    delay: float = 7,
    output_prefix: str | None = None,
    uncomment: bool = False,
) -> dict[str, pd.DataFrame]:
    collected: dict[str, list[pd.DataFrame]] = {name: [] for name in tables}
    missing_report: dict[str, list[str]] = {}  # season -> missing table names
    failed_seasons: list[str] = []
    last = len(season_urls) - 1

    for i, (season, url) in enumerate(season_urls.items()):
        print(f"Scraping season: {season}...")
        try:
            found, missing = scrape_page_tables(driver, url, tables, season=season,  uncomment=uncomment)
        except Exception as e:  # the whole page failed (blocked, timeout, etc.)
            print(f"  Page failed {season}: {type(e).__name__}: {e}")
            print(f"  Page title was: {driver.title!r}")
            with open(f"debug_{season}.html", "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            failed_seasons.append(season)
            continue

        for name, df in found.items():
            collected[name].append(df.assign(Season=season))

        if missing:
            missing_report[season] = missing
            print(f"  {season}: got {list(found)}, missing {missing}")
        else:
            print(f"  {season} done.")

        if i < last:
            time.sleep(delay)

    results = {
        name: pd.concat(dfs, ignore_index=True)
        for name, dfs in collected.items()
        if dfs
    }

    if output_prefix:
        seasons = list(season_urls)
        span = f"{seasons[0][:4]}_{seasons[-1][-4:]}"
        for name, df in results.items():
            path = f"{output_prefix}_{name}_{span}.csv"
            df.to_csv(path, index=False)
            print(f"Saved {name} -> {path}")

    # Summary so nothing is silently lost
    print("\n--- Summary ---")
    for name in tables:
        have = collected[name]
        print(f"{name}: {len(have)}/{len(season_urls)} seasons")
    if missing_report:
        print("Missing tables:", missing_report)
    if failed_seasons:
        print("Failed pages:", failed_seasons)

    return results

In [7]:
driver = uc.Chrome()

In [13]:
attrs = {
    "rank_overall": {"id": "results{season}91_overall"},
    "rank_home_away": {"id": "results{season}91_home_away"},
    "squad_stats": {"id": "stats_squads_standard_for"},
    "opponent_stats": {"id": "stats_squads_standard_against"},
    "squad_goalkeeping": {"id": "stats_squads_keeper_for"},
    "oppenent_goalkeeping": {"id": "stats_squads_keeper_against"},
    "squad_shooting": {"id": "stats_squads_shooting_for"},
    "oppenent_shooting": {"id": "stats_squads_shooting_against"},
    "squad_playingtime": {"id": "stats_squads_playing_time_for"},
    "oppenent_playingtime": {"id": "stats_squads_playing_time_against"},
    "squad_misc": {"id": "stats_squads_misc_for"},
    "oppenent_misc": {"id": "stats_squads_misc_against"},
}

In [14]:
season_urls = dict(zip(seasons, urls))

data = scrape_seasons(
    driver,
    season_urls,
    tables=attrs,
    output_prefix="epl",
)

Scraping season: 2021-2022...
  2021-2022 done.
Scraping season: 2022-2023...
  2022-2023 done.
Scraping season: 2023-2024...
  2023-2024 done.
Scraping season: 2024-2025...
  2024-2025 done.
Scraping season: 2025-2026...
  2025-2026 done.
Saved rank_overall -> epl_rank_overall_2021_2026.csv
Saved rank_home_away -> epl_rank_home_away_2021_2026.csv
Saved squad_stats -> epl_squad_stats_2021_2026.csv
Saved opponent_stats -> epl_opponent_stats_2021_2026.csv
Saved squad_goalkeeping -> epl_squad_goalkeeping_2021_2026.csv
Saved oppenent_goalkeeping -> epl_oppenent_goalkeeping_2021_2026.csv
Saved squad_shooting -> epl_squad_shooting_2021_2026.csv
Saved oppenent_shooting -> epl_oppenent_shooting_2021_2026.csv
Saved squad_playingtime -> epl_squad_playingtime_2021_2026.csv
Saved oppenent_playingtime -> epl_oppenent_playingtime_2021_2026.csv
Saved squad_misc -> epl_squad_misc_2021_2026.csv
Saved oppenent_misc -> epl_oppenent_misc_2021_2026.csv

--- Summary ---
rank_overall: 5/5 seasons
rank_home_a

## Players Data

In [20]:
def create_urls(start_year: int, end_year:int):
    seasons = [f"{year}-{year+1}" for year in range(start_year, end_year)]

    urls = []
    
    for season in seasons:
        urls.append(f"https://fbref.com/en/comps/9/{season}/shooting/{season}-Premier-League-Stats")

    return urls, seasons

In [21]:
urls, seasons = create_urls(2021, 2026)

In [22]:
attrs = {
    "players_shooting": {"id": "stats_shooting"}
}

In [23]:
season_urls = dict(zip(seasons, urls))

data = scrape_seasons(
    driver,
    season_urls,
    tables=attrs,
    output_prefix="epl",
)

Scraping season: 2021-2022...
  2021-2022 done.
Scraping season: 2022-2023...
  2022-2023 done.
Scraping season: 2023-2024...
  2023-2024 done.
Scraping season: 2024-2025...
  2024-2025 done.
Scraping season: 2025-2026...
  2025-2026 done.
Saved players_shooting -> epl_players_shooting_2021_2026.csv

--- Summary ---
players_shooting: 5/5 seasons


In [2]:
import argparse
import time
from pathlib import Path

import requests

In [4]:
BASE_URL = "https://football-data.co.uk/mmz4281/{season}/{league}.csv"
LEAGUES = ["E0"]  # Premier League, Championship, L1, L2, Conference
HEADERS = {"User-Agent": "Mozilla/5.0 (football-data downloader)"}

In [5]:
def season_code(start_year: int) -> str:
    """2025 -> '2526', 1999 -> '9900'."""
    return f"{start_year % 100:02d}{(start_year + 1) % 100:02d}"


In [6]:
def season_label(start_year: int) -> str:
    """2025 -> '2025-2026'."""
    return f"{start_year}-{start_year + 1}"

In [7]:
def download_file(url: str, dest: Path, overwrite: bool = False) -> str:
    if dest.exists() and not overwrite:
        return "skipped (exists)"
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
    except requests.RequestException as e:
        return f"failed ({e})"
    if r.status_code != 200:
        return f"failed (HTTP {r.status_code})"
    dest.write_bytes(r.content)
    return "downloaded"

In [8]:
def download_season(start_year: int, out_dir: Path, overwrite: bool = False) -> list[pd.DataFrame]:
    """Download one season's league files and return them as DataFrames."""
    code = season_code(start_year)
    label = season_label(start_year)
    season_dir = out_dir / code
    season_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    for league in LEAGUES:
        url = BASE_URL.format(season=code, league=league)
        dest = season_dir / f"{league}.csv"
        status = download_file(url, dest, overwrite)
        print(f"[{label}] {league}: {status}")
        time.sleep(0.5)

        if not dest.exists():
            continue
        try:
            # football-data files are often latin-1 encoded
            df = pd.read_csv(dest, encoding="latin-1")
        except Exception as e:
            print(f"[{label}] {league}: could not read file ({e})")
            continue

        # Drop empty trailing rows/columns that appear in some seasons
        df = df.dropna(how="all")
        df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

        df.insert(0, "Season", label)
        df.insert(1, "League", league)
        frames.append(df)
    return frames

In [9]:
def build_matches(start_years, raw_dir: Path, matches_path: Path, overwrite: bool = False) -> pd.DataFrame:
    all_frames = []
    for year in start_years:
        all_frames.extend(download_season(year, raw_dir, overwrite))

    if not all_frames:
        raise RuntimeError("No data was downloaded or found.")

    # Columns differ between seasons; concat aligns them and fills gaps with NaN
    matches = pd.concat(all_frames, ignore_index=True, sort=False)

    matches_path.parent.mkdir(parents=True, exist_ok=True)
    matches.to_csv(matches_path, index=False)
    print(f"\nSaved {len(matches):,} matches to {matches_path}")
    return matches

In [10]:
build_matches(
        start_years=range(2021, 2026),
        raw_dir=Path("../data/raw/"),
        matches_path=Path("../data/Matches.csv"),
    )

[2021-2022] E0: downloaded
[2022-2023] E0: downloaded
[2023-2024] E0: downloaded
[2024-2025] E0: downloaded


C:\Users\Admin\AppData\Local\Temp\ipykernel_9844\2191108573.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, "Season", label)
C:\Users\Admin\AppData\Local\Temp\ipykernel_9844\2191108573.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(1, "League", league)


[2025-2026] E0: downloaded


C:\Users\Admin\AppData\Local\Temp\ipykernel_9844\2191108573.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(0, "Season", label)
C:\Users\Admin\AppData\Local\Temp\ipykernel_9844\2191108573.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df.insert(1, "League", league)



Saved 1,900 matches to ..\data\Matches.csv


,Season,League,ï»¿Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,2021-2022,E0,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-2022,E0,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-2022,E0,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-2022,E0,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-2022,E0,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1895,2025-2026,E0,E0,24/05/2026,16:00,Man City,Aston Villa,1,2,A,...,6.75,1.4,5.25,7.00,1.40,5.25,6.00,1.40,5.25,6.00
1896,2025-2026,E0,E0,24/05/2026,16:00,Nott'm Forest,Bournemouth,1,1,D,...,1.90,3.5,3.80,1.95,3.40,3.60,2.05,3.40,3.60,2.05
1897,2025-2026,E0,E0,24/05/2026,16:00,Sunderland,Chelsea,2,1,H,...,1.96,3.4,3.70,2.05,3.40,3.60,2.05,3.40,3.50,2.05
1898,2025-2026,E0,E0,24/05/2026,16:00,Tottenham,Everton,1,0,H,...,4.10,1.9,3.40,4.33,1.85,3.50,4.20,1.85,3.50,4.00
